In [2]:
import pandas as pd
import csv
import json
from openai import OpenAI
import time


In [4]:
import tiktoken
import os

In [ ]:
muj_api_klic = ""
client = OpenAI(api_key=muj_api_klic)

In [21]:
#df1 = pd.read_csv('duvody_v_layoffs_14_05.csv')
#df1

In [22]:
#sloupce = ["index", "text", "odpoved"]
#df_sloupce = df1[sloupce]

In [23]:
#df_cista = df_sloupce[
    #(df_sloupce["odpoved"] == "YES") &
    #((df_sloupce["text"].notna()))
#]
#df_cista.to_csv('testovani_promptu.csv')
#df_cista

In [24]:
#df_je_tam_duvod = pd.read_csv('je_tam_duvod_vypovedi.csv')
#df_ocisteny = pd.read_csv('ocisteny_layoffs.csv')
#df_ocisteny

In [ ]:
df1 = pd.read_csv('duvody_v_layoffs_14_05.csv')

# prázdný list, kam si budu ukládat odpovědi chatu
results = []
results_path = "zkracovani.csv"

if os.path.exists(results_path):
    existing_results = pd.read_csv(results_path)
    processed_ids = set(existing_results['id'].tolist())
    results = existing_results.to_dict(orient='records')
    print(f"Načteno {len(results)} již zpracovaných výsledků.")
else:
    processed_ids = set()
    print("Nebyl nalezen žádný předchozí výstupní soubor. Začíná se od začátku.")
# Filtrování nezpracovaných řádků
df_to_process = df1[~df1['index'].isin(processed_ids)]


for index, row in df1.iterrows():
    index_clanku = row['index']
    text_clanku   = row['text']
    odpoved       = row['odpoved']
    enc = tiktoken.encoding_for_model("gpt-4o")
    num_tokens = len(enc.encode(text_clanku))
    print("Tokenů:", num_tokens)
    if num_tokens < 150000:

    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system",
            "content": (
                "You are an assistant helping with summarizing news articles about layoffs. "
                "Your task is to create a concise summary of the given article that contains all "
                "relevant information about layoffs."
            )},
            {"role": "user",
            "content": (
                "Summarize the following article in no more than 15 sentences. "
                "Keep all relevant information related to layoffs, including:\n"
                "- the company name,\n"
                "- number of jobs affected,\n"
                "- reasons or motivations for the layoffs,\n"
                "- timing or date,\n"
                "- affected departments or regions,\n"
                "- any management quotes or company statements,\n"
                "- relevant market or financial context.\n\n"
                "Article:\n" + text_clanku
            )}
        ]
    )

    zkraceny_text = completion.choices[0].message.content
    time.sleep(60)
    # uložíme výsledek do listu
    results.append({
        'id':      index_clanku,
        'zkraceny': zkraceny_text,      # <-- tady chyběla čárka
        'odpoved':  odpoved
    })

    # průběžné ukládání každých 5 článků
    if (index + 1) % 5 == 0:
        df_results = pd.DataFrame(results)
        df_results.to_csv("zkracovani_prubezne.csv", index=False)
        print(f"Průběžně uloženo {len(results)} výsledků do souboru.")

# finální uložení po skončení smyčky
df_results = pd.DataFrame(results)
df_results.to_csv("zkracovani.csv", index=False)
print("Finální uložení hotovo.")

Průběžně uloženo 5 výsledků do souboru.
Průběžně uloženo 10 výsledků do souboru.
Průběžně uloženo 15 výsledků do souboru.
Průběžně uloženo 20 výsledků do souboru.
Průběžně uloženo 25 výsledků do souboru.
Průběžně uloženo 30 výsledků do souboru.


TypeError: can only concatenate str (not "float") to str

In [ ]:
df_ukazka_vysledku = pd.read_csv('zkracovani_komplet_15_05.csv')
df_ukazka_vysledku ['odpoved'].value_counts()


odpoved
YES    2450
NO      910
Name: count, dtype: int64

In [31]:
df_ukaz_co_mas = pd.read_csv('zkracovani_prubezne.csv')
df_ukaz_co_mas

,id,zkraceny,odpoved
0,0,AI proptech startup Tract has officially shut ...,YES
1,1,"Automattic, the parent company of WordPress.co...",YES
2,2,Canva has announced layoffs affecting the majo...,YES
3,3,Data analytics startup WhyHive is shutting dow...,YES
4,4,Swedish battery-maker Northvolt has laid off m...,YES
...,...,...,...
65,69,Tripadvisor has announced the termination of m...,YES
66,70,Intel Corp. has announced it will lay off an a...,YES
67,71,"Salesforce is cutting over 1,000 jobs as it em...",YES
68,72,"Sure Insurance, an embedded insurtech company,...",YES


In [9]:
import httpx
from openai import OpenAI



# 1) Načtení dat
df1 = pd.read_csv('duvody_v_layoffs_14_05.csv')
df1['text'] = df1['text'].fillna('').astype(str)

# 2) Příprava výstupů
results = []
results_path = "zkracovani_prubezene.csv"

if os.path.exists(results_path):
    existing_results = pd.read_csv(results_path)
    processed_ids = set(existing_results['id'].tolist())
    results = existing_results.to_dict(orient='records')
    print(f"Načteno {len(results)} již zpracovaných výsledků.")
else:
    processed_ids = set()
    print("Nebyl nalezen žádný předchozí výstupní soubor. Začíná se od začátku.")

# 3) Vyfiltrování nových článků
df_to_process = df1[~df1['index'].isin(processed_ids)]

# 4) Smyčka přes nové články
for _, row in df_to_process.iterrows():
    index_clanku = row['index']
    text_clanku  = row['text'].strip()
    odpoved      = row.get('odpoved', '')

    if not text_clanku:
        continue

    enc = tiktoken.encoding_for_model("gpt-4o-mini")
    num_tokens = len(enc.encode(text_clanku))
    print(f"Článek {index_clanku} – tokenů: {num_tokens}")
    if num_tokens >= 150_000:
        print(f"Článek {index_clanku} je příliš dlouhý, přeskočeno.")
        continue

    prompt = (
        "Summarize the following article in no more than 15 sentences. "
        "Keep all relevant information related to layoffs, including:\n"
        "- the company name,\n"
        "- number of jobs affected,\n"
        "- reasons or motivations for the layoffs,\n"
        "- timing or date,\n"
        "- affected departments or regions,\n"
        "- any management quotes or company statements,\n"
        "- relevant market or financial context.\n\n"
        f"Article:\n{text_clanku}"
    )

    # retry logic pro timeout via httpx
    max_retries = 3
    zkraceny_text = ""
    for attempt in range(1, max_retries + 1):
        try:
            completion = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are an assistant helping with summarizing news articles about layoffs. "
                            "Your task is to create a concise summary of the given article that contains all "
                            "relevant information about layoffs."
                        )
                    },
                    {"role": "user", "content": prompt}
                ],
                timeout=30
            )
            zkraceny_text = completion.choices[0].message.content
            break
        except httpx.TimeoutException:
            print(f"[{index_clanku}] Pokus {attempt}/{max_retries} – timeout, zkouším znovu...")
            if attempt < max_retries:
                time.sleep(5)
            else:
                print(f"[{index_clanku}] Timeout i po {max_retries} pokusech, přeskočeno.")
        except Exception as e:
            print(f"[{index_clanku}] Neočekávaná chyba: {e}, přeskočeno.")
            break

    # pauza kvůli rate limitu
    time.sleep(1)

    # uložení výsledku
    results.append({
        'id':       index_clanku,
        'zkraceny': zkraceny_text,
        'odpoved':  odpoved
    })

    # průběžné ukládání každých 5 článků
    if len(results) % 5 == 0:
        pd.DataFrame(results).to_csv("zkracovani_prubezne.csv", index=False)
        print(f"Průběžně uloženo {len(results)} výsledků do zkracovani_prubezne.csv.")

# 5) Finální uložení
pd.DataFrame(results).to_csv(results_path, index=False)
print(f"Finální uložení hotovo do {results_path}.")




Nebyl nalezen žádný předchozí výstupní soubor. Začíná se od začátku.
Článek 0 – tokenů: 376
Článek 1 – tokenů: 836
Článek 2 – tokenů: 399
Článek 3 – tokenů: 820
Článek 4 – tokenů: 535
Průběžně uloženo 5 výsledků do zkracovani_prubezne.csv.
Článek 5 – tokenů: 523
Článek 6 – tokenů: 369
Článek 7 – tokenů: 1203
Článek 8 – tokenů: 683
Článek 9 – tokenů: 29
Průběžně uloženo 10 výsledků do zkracovani_prubezne.csv.
Článek 10 – tokenů: 399
Článek 11 – tokenů: 18
Článek 12 – tokenů: 750
Článek 13 – tokenů: 456
Článek 14 – tokenů: 255
Průběžně uloženo 15 výsledků do zkracovani_prubezne.csv.
Článek 15 – tokenů: 256
Článek 16 – tokenů: 407
Článek 17 – tokenů: 31
Článek 18 – tokenů: 506
Článek 19 – tokenů: 42
Průběžně uloženo 20 výsledků do zkracovani_prubezne.csv.
Článek 20 – tokenů: 289
Článek 21 – tokenů: 162
Článek 22 – tokenů: 710
Článek 23 – tokenů: 1939
Článek 24 – tokenů: 870
Průběžně uloženo 25 výsledků do zkracovani_prubezne.csv.
Článek 25 – tokenů: 468
Článek 26 – tokenů: 111
Článek 27 –